# Session 2: Chains & Memory (50 minutes)

## 🎯 Learning Objectives
- Understand why chains are needed
- Build sequential multi-step workflows
- Implement conversation memory
- Create a chatbot that remembers context

## 📋 Problem Statement
Our Research Assistant needs to:
- Execute multi-step research tasks
- Remember previous conversation context

## ⏱️ Session Breakdown
- 5 min: The problem with single calls
- 15 min: Chains deep dive
- 15 min: Memory types and implementation
- 10 min: Building the chatbot
- 5 min: Recap

---

## 🖥️ Using TinyLlama via Ollama (LOCAL, FREE, NO LIMITS!)

## 1. Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "false"

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize TinyLlama via Ollama (LOCAL, FREE!)
llm = ChatOllama(
    model="qwen2:0.5b",
    temperature=0.7
)

print("✅ Session 2 Setup Complete!")
print("🖥️  Using qwen2:0.5b via Ollama - LOCAL, FREE, NO LIMITS!")

## 2. The Problem: LLMs Have No Memory!

🎯 **Problem**: Each LLM call is STATELESS - the model has no memory!

In [ ]:
# First message
response1 = llm.invoke("My name is Alice and I'm learning LangChain.")
print("📝 Response 1:", response1.content)

# Second message - LLM doesn't remember!
response2 = llm.invoke("What's my name?")
print("📝 Response 2:", response2.content)
print("\n❌ The LLM doesn't remember your name!")

## 3. The Problem: Complex Tasks Need Multiple Steps

Single LLM calls can't handle complex, multi-step tasks.

Example: "Research quantum computing and create a beginner-friendly summary"

This requires:
1. Research/gather information
2. Analyze and extract key points
3. Simplify for beginners
4. Format the output

## 4. Simple Chain - Connecting Components

In [ ]:
# Create a simple chain
simple_chain = (
    ChatPromptTemplate.from_template("Explain {topic} in one paragraph.")
    | llm
    | StrOutputParser()
)

result = simple_chain.invoke({"topic": "neural networks"})
print("🔗 Simple Chain Result:\n", result)

## 5. Sequential Chains - Multi-Step Processing

Output of one step becomes input to the next.

**Research Assistant Example:**
- Step 1: Research the topic → Technical content
- Step 2: Simplify the content → Beginner-friendly version
- Step 3: Create quiz questions → Test understanding

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Step 1: Research Chain
research_prompt = ChatPromptTemplate.from_template(
    """You are a research expert. Provide detailed technical information about:
    Topic: {topic}
    
    Include key concepts, history, and current applications."""
)
research_chain = research_prompt | llm | StrOutputParser()

# Step 2: Simplify Chain
simplify_prompt = ChatPromptTemplate.from_template(
    """Take this technical content and rewrite it for a complete beginner.
    Use simple words and analogies.
    
    Technical Content:
    {research_output}"""
)
simplify_chain = simplify_prompt | llm | StrOutputParser()

# Step 3: Quiz Chain
quiz_prompt = ChatPromptTemplate.from_template(
    """Based on this content, create 3 simple quiz questions to test understanding.
    
    Content:
    {simplified_content}"""
)
quiz_chain = quiz_prompt | llm | StrOutputParser()

## 6. Connecting the Sequential Chain

In [ ]:
def process_research(input_dict):
    """Run research and return result"""
    topic = input_dict["topic"]
    research = research_chain.invoke({"topic": topic})
    return {"research_output": research, "topic": topic}

def process_simplify(input_dict):
    """Simplify the research"""
    simplified = simplify_chain.invoke({"research_output": input_dict["research_output"]})
    return {"simplified_content": simplified, **input_dict}

def create_quiz(input_dict):
    """Create quiz from simplified content"""
    quiz = quiz_chain.invoke({"simplified_content": input_dict["simplified_content"]})
    return {"quiz": quiz, **input_dict}

# Complete sequential chain
full_research_chain = (
    RunnableLambda(process_research)
    | RunnableLambda(process_simplify)
    | RunnableLambda(create_quiz)
)

# Execute the full chain
print("🔄 Running full research chain...")
result = full_research_chain.invoke({"topic": "Machine Learning"})

print("\n📚 RESEARCH OUTPUT (excerpt):")
print(result["research_output"][:300] + "...")

print("\n📖 SIMPLIFIED VERSION (excerpt):")
print(result["simplified_content"][:300] + "...")

print("\n❓ QUIZ QUESTIONS:")
print(result["quiz"])

## 7. Parallel Chains - Run Multiple Analyses

Use `RunnableParallel` to run multiple chains simultaneously.

In [ ]:
from langchain_core.runnables import RunnableParallel

# Run multiple analyses in parallel
parallel_analysis = RunnableParallel(
    summary=ChatPromptTemplate.from_template(
        "Summarize this topic in 2 sentences: {topic}"
    ) | llm | StrOutputParser(),
    
    applications=ChatPromptTemplate.from_template(
        "List 3 real-world applications of: {topic}"
    ) | llm | StrOutputParser(),
    
    challenges=ChatPromptTemplate.from_template(
        "What are 2 main challenges with: {topic}"
    ) | llm | StrOutputParser()
)

# Execute parallel chains
print("⚡ Running parallel analysis...")
parallel_result = parallel_analysis.invoke({"topic": "Artificial Intelligence"})

print("\n📊 PARALLEL RESULTS:")
print(f"Summary: {parallel_result['summary']}")
print(f"\nApplications: {parallel_result['applications']}")
print(f"\nChallenges: {parallel_result['challenges']}")

## 8. Memory: Making LLMs Remember

🎯 **Problem**: LLMs are stateless. They forget everything between calls.

💡 **Solution**: Memory - We store and inject conversation history!

### Memory Types:
| Type | Use Case | Pros | Cons |
|------|----------|------|------|
| ConversationBufferMemory | Short chats | Full context | Token explosion |
| ConversationSummaryMemory | Long chats | Efficient | Loses detail |
| ConversationBufferWindowMemory | Balanced | Predictable | Fixed window |

## 9. Conversation Buffer Memory

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# Simple in-memory chat history storage
class SimpleMemory:
    def __init__(self):
        self.messages = []
    
    def save_context(self, inputs: dict, outputs: dict):
        """Save user input and AI output"""
        self.messages.append(HumanMessage(content=inputs["input"]))
        self.messages.append(AIMessage(content=outputs["output"]))
    
    def load_memory_variables(self, inputs: dict = None) -> dict:
        """Load conversation history"""
        return {"chat_history": self.messages}

# Create memory
buffer_memory = SimpleMemory()

# Manually add some conversation history
buffer_memory.save_context(
    {"input": "Hi, I'm learning about AI"},
    {"output": "Hello! That's great. AI is a fascinating field. What aspect interests you most?"}
)
buffer_memory.save_context(
    {"input": "I'm interested in natural language processing"},
    {"output": "NLP is a great choice! It's the technology behind chatbots, translation, and more."}
)

# Load the memory
history = buffer_memory.load_memory_variables({})
print("📝 Buffer Memory Contents:")
for msg in history["chat_history"]:
    role = "Human" if isinstance(msg, HumanMessage) else "AI"
    print(f"  {role}: {msg.content[:60]}...")

## 10. Conversation Summary Memory

In [ ]:
# For this demo, we'll use a simple summary memory
class SimpleSummaryMemory:
    def __init__(self):
        self.summary = ""
        self.messages = []
    
    def save_context(self, inputs: dict, outputs: dict):
        """Save and summarize"""
        self.messages.append((inputs["input"], outputs["output"]))
        # For demo, just concatenate key points
        self.summary += f"User: {inputs['input'][:40]}...\nAI: {outputs['output'][:40]}...\n\n"
    
    def load_memory_variables(self, inputs: dict = None) -> dict:
        """Load conversation summary"""
        return {"chat_history": self.summary if self.summary else "No history yet"}

# Create summary memory
summary_memory = SimpleSummaryMemory()

# Add conversations
summary_memory.save_context(
    {"input": "I want to learn about transformers architecture"},
    {"output": "Transformers are neural networks that use self-attention mechanisms. They were introduced in the 'Attention is All You Need' paper."}
)
summary_memory.save_context(
    {"input": "How do they compare to RNNs?"},
    {"output": "Unlike RNNs, transformers process all tokens in parallel, making them faster. They also handle long-range dependencies better through attention."}
)

# Load - notice it's a summary, not full history
history = summary_memory.load_memory_variables({})
print("📝 Summary Memory Contents:")
print(f"  {history['chat_history']}")

## 11. Conversation Buffer Window Memory

In [ ]:
# Window memory keeps only last K interactions
class SimpleWindowMemory:
    def __init__(self, k: int = 2):
        self.k = k
        self.messages = []
    
    def save_context(self, inputs: dict, outputs: dict):
        """Save and maintain window"""
        self.messages.append(HumanMessage(content=inputs["input"]))
        self.messages.append(AIMessage(content=outputs["output"]))
        # Keep only last 2K messages
        if len(self.messages) > self.k * 2:
            self.messages = self.messages[-(self.k * 2):]
    
    def load_memory_variables(self, inputs: dict = None) -> dict:
        """Load recent messages only"""
        return {"chat_history": self.messages}

# Keep only last 2 interactions
window_memory = SimpleWindowMemory(k=2)

# Add 3 conversations - only last 2 will be kept
window_memory.save_context({"input": "Message 1"}, {"output": "Response 1"})
window_memory.save_context({"input": "Message 2"}, {"output": "Response 2"})
window_memory.save_context({"input": "Message 3"}, {"output": "Response 3"})

history = window_memory.load_memory_variables({})
print("📝 Window Memory (k=2):")
for msg in history["chat_history"]:
    print(f"  {msg.content}")  # Only shows Message 2, 3 and their responses

## 12. Building a Chatbot with Memory

In [ ]:
from langchain_core.prompts import MessagesPlaceholder

# Create the prompt with history placeholder
chatbot_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful AI research assistant. Your capabilities:
    - Answer questions about technology, science, and AI
    - Remember our conversation context
    - Provide accurate, well-researched information
    - Ask clarifying questions when needed"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Initialize memory using our simple implementation
chat_memory = SimpleMemory()

## 13. The Chatbot Function

In [ ]:
def chat(user_input: str) -> str:
    """
    Process user input and return AI response with memory.
    """
    # Load chat history
    history = chat_memory.load_memory_variables({})
    
    # Create chain
    chain = chatbot_prompt | llm | StrOutputParser()
    
    # Get response
    response = chain.invoke({
        "chat_history": history["chat_history"],
        "input": user_input
    })
    
    # Save to memory (ensure response is string)
    chat_memory.save_context(
        {"input": user_input},
        {"output": str(response)}
    )
    
    return response

## 14. Test the Chatbot

In [ ]:
print("🤖 CHATBOT WITH MEMORY DEMO")
print("=" * 50)

# First interaction - introduce yourself
response1 = chat("Hi! My name is Alex and I'm learning about machine learning.")
print(f"You: Hi! My name is Alex and I'm learning about machine learning.")
print(f"Bot: {response1}")
print()

# Second interaction - ask about something
response2 = chat("What's the difference between supervised and unsupervised learning?")
print(f"You: What's the difference between supervised and unsupervised learning?")
print(f"Bot: {response2}")
print()

# Third interaction - test if it remembers name
response3 = chat("By the way, what's my name?")
print(f"You: By the way, what's my name?")
print(f"Bot: {response3}")
print()

# Fourth interaction - build on previous topic
response4 = chat("Which type would be better for clustering customer data?")
print(f"You: Which type would be better for clustering customer data?")
print(f"Bot: {response4}")

print("\n✅ The chatbot remembers both your name AND the ML discussion!")

## 15. View the Memory Contents

In [ ]:
print("\n📝 MEMORY CONTENTS:")
print("=" * 50)
history = chat_memory.load_memory_variables({})
for i, msg in enumerate(history["chat_history"]):
    role = "Human" if isinstance(msg, HumanMessage) else "AI"
    print(f"{i+1}. [{role}]: {msg.content[:80]}...")

## 16. Modern Approach: RunnableWithMessageHistory

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# Store for multiple sessions
session_store = {}

def get_session_history(session_id: str):
    """Get or create message history for a session."""
    if session_id not in session_store:
        session_store[session_id] = ChatMessageHistory()
    return session_store[session_id]

# Create the base chain
base_chain = chatbot_prompt | llm | StrOutputParser()

# Wrap with message history
chain_with_history = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# Use with session ID
config = {"configurable": {"session_id": "user-123"}}

response = chain_with_history.invoke(
    {"input": "I'm interested in NLP"}, 
    config=config
)
print("\n🆕 Modern Memory Approach:")
print(f"Response: {response}")

## 📚 Session 2 Recap

### Key Takeaways:

1. **CHAINS** connect components for multi-step workflows
   - Simple: `prompt | llm | parser`
   - Sequential: Step1 → Step2 → Step3
   - Parallel: Run multiple chains simultaneously

2. **MEMORY** makes LLMs stateful
   - Buffer: Store everything (simple, grows large)
   - Summary: Compress history (efficient)
   - Window: Keep last K (predictable)

3. **Building Chatbots:**
   - Use `MessagesPlaceholder` for history
   - Save context after each interaction
   - Modern: `RunnableWithMessageHistory`

4. **The pattern:**
   ```
   Load History → Generate Response → Save to Memory
   ```

---

### 🔜 Next Session: RAG - Document Q&A
"The LLM remembers our chat, but still doesn't know about MY documents!"

In [ ]:
print("""
╔═══════════════════════════════════════════════════════════════════════════╗
║                    SESSION 2 COMPLETE! 🎉                                  ║
║                                                                            ║
║  ☕ BREAK TIME - 10 MINUTES ☕                                              ║
║                                                                            ║
║  Next: Session 3 - RAG (Retrieval Augmented Generation)                    ║
║  File: 03_rag_implementation.ipynb                                         ║
║                                                                            ║
║  "Now we have memory... but what about YOUR documents?"                    ║
║  "The LLM still can't read your PDFs, websites, or databases!"             ║
╚═══════════════════════════════════════════════════════════════════════════╝
""")